## The push/pull workflow

Getting an image you built onto a registry is a four-step cycle — **tag, login, push, pull**:

```bash
# 1. Tag your local image with the registry-qualified name
docker tag flask-demo:0.1 ghcr.io/myorg/flask-demo:0.1

# 2. Authenticate
docker login ghcr.io

# 3. Push
docker push ghcr.io/myorg/flask-demo:0.1

# 4. (elsewhere) Pull and run
docker pull ghcr.io/myorg/flask-demo:0.1
docker run ghcr.io/myorg/flask-demo:0.1
```

**What `docker tag` does:** *nothing to the data.* It adds another **name pointing at the same image** — same layers, same digest. That's why you can tag one build as `:0.1` and `:latest` and push both for the cost of one upload; they resolve to the same manifest.

**What actually uploads on `push`.** The daemon walks the image's layers and, for each, asks the registry *"do you have `sha256:abc…`?"* (a HEAD request). If yes, it **skips**; if no, it streams the layer. So a new build sharing 90% of its layers with the last one uploads only the changed 10% — the **content-addressable dedup** from the previous section, in action. It's why `push` is often far quicker than the image's total size implies.

The key mental correction here: **a registry name is part of the image reference, not a separate destination.** You don't "push to a place" so much as *rename* the image with a registry-qualified name and then push that name. `ghcr.io/myorg/flask-demo:0.1` *is* the image's identity on GHCR. Get the tag right and `push`/`pull` are trivial; get it wrong (missing namespace, wrong host) and you'll get a confusing auth or not-found error — the reference, not the command, is almost always the thing to check.